# LSTM + GNN Combined Model
**Phase 4 — F1 Deep Learning Project**

Spatio-Temporal model combining LSTM (temporal) + GNN (spatial) branches.

In [1]:
# Cell 1 — Imports
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error
import matplotlib.pyplot as plt
import joblib, sys
sys.path.append('..')
from models.lstm_gnn_model import LSTM_GNN
from models.gnn_model import build_adjacency
print('Imports done!')

Imports done!


In [2]:
# Cell 2 — Load Data
df = pd.read_csv('../f1.csv', index_col=0).dropna()

SEQ_LEN  = 5
features = ['lap','position','pit_stop','tyre_age','grid',
            'alt','driver_skill','circuit_difficulty',
            'race_year','round','prev_lap_time']

print('Data loaded:', df.shape)

Data loaded: (372493, 19)


C:\Users\Asus'\AppData\Local\Temp\ipykernel_20908\3348116256.py:2: DtypeWarning: Columns (0: positionOrder) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../f1.csv', index_col=0).dropna()


In [3]:
# Cell 3 — Build Sequences + Graphs
# For each lap: create LSTM sequence + GNN graph
samples = []

for (race_id, lap_num), lap_grp in df.groupby(['raceId','lap']):
    if len(lap_grp) < 3:
        continue
    # GNN data: all drivers in this lap
    node_feats = torch.FloatTensor(lap_grp[features].values)
    positions  = lap_grp['position'].tolist()
    adj        = build_adjacency(positions, threshold=5)
    targets    = lap_grp['lap_time_ms'].values

    # LSTM data: get previous SEQ_LEN laps for each driver
    for i, (_, row) in enumerate(lap_grp.iterrows()):
        driver_id = row['driverId']
        driver_hist = df[(df['raceId']==race_id) &
                         (df['driverId']==driver_id) &
                         (df['lap'] < lap_num)].tail(SEQ_LEN)
        if len(driver_hist) < SEQ_LEN:
            continue
        seq = torch.FloatTensor(driver_hist[features].values).unsqueeze(0)
        samples.append((seq, node_feats, adj, torch.tensor([targets[i]], dtype=torch.float)))

print(f'Total training samples: {len(samples)}')

KeyError: "['driver_skill', 'circuit_difficulty', 'prev_lap_time'] not in index"

In [ ]:
# Cell 4 — Train LSTM+GNN
model     = LSTM_GNN(lstm_input=11, lstm_hidden=128, gnn_input=11, gnn_hidden=64)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn   = nn.MSELoss()

split        = int(0.8 * len(samples))
train_data   = samples[:split]
test_data    = samples[split:]

train_losses = []
EPOCHS = 20

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    for seq, node_feats, adj, target in train_data:
        pred       = model(seq, node_feats, adj)
        loss       = loss_fn(pred, target)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    avg = epoch_loss / len(train_data)
    train_losses.append(avg)
    print(f'Epoch {epoch+1}/{EPOCHS} | Loss: {avg:.2f}')

torch.save(model.state_dict(), '../lstm_gnn_model.pth')
print('LSTM+GNN model saved!')

In [ ]:
# Cell 5 — Evaluate
model.eval()
all_preds, all_targets = [], []

with torch.no_grad():
    for seq, node_feats, adj, target in test_data:
        pred = model(seq, node_feats, adj)
        all_preds.append(pred.item())
        all_targets.append(target.item())

r2  = r2_score(all_targets, all_preds)
mae = mean_absolute_error(all_targets, all_preds) / 1000
print(f'LSTM+GNN R2  : {r2:.4f}')
print(f'LSTM+GNN MAE : {mae:.3f} seconds')

In [ ]:
# Cell 6 — Plot Training Loss
plt.figure(figsize=(10,5))
plt.plot(train_losses, color='green', linewidth=2)
plt.title('LSTM+GNN Combined Training Loss', fontsize=14)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.grid(True)
plt.tight_layout()
plt.savefig('../lstm_gnn_training_loss.png')
plt.show()